# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [11]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [6]:
# A.1. Data size, column names, data types
print('Kích thước dữ liệu:', df.shape)
print('Tên cột:', list(df.columns))
print('\nThông tin kiểu dữ liệu:')
df.info()

Kích thước dữ liệu: (541909, 8)
Tên cột: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Thông tin kiểu dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str           
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  str           
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 33.1 MB


## A.2. Missing values & Duplicate data

In [7]:
# A.2. Missing values & Duplicate data
print('Giá trị thiếu:\n', df.isnull().sum())
print('\nSố dòng trùng lặp:', df.duplicated().sum())
print('Số dòng sau khi bỏ trùng:', df.drop_duplicates().shape[0])

Giá trị thiếu:
 InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Số dòng trùng lặp: 5268
Số dòng sau khi bỏ trùng: 536641


## A.3. Invalid values

In [9]:
# A.3. Invalid values
print('Số Quantity < 0:', (df['Quantity'] < 0).sum())
print('Số UnitPrice <= 0:', (df['UnitPrice'] <= 0).sum())
print('Số CustomerID thiếu:', df['CustomerID'].isnull().sum())
print('\nMô tả nhanh của Quantity và UnitPrice:\n')
print(df[['Quantity', 'UnitPrice']].describe())

Số Quantity < 0: 10624
Số UnitPrice <= 0: 2517
Số CustomerID thiếu: 135080

Mô tả nhanh của Quantity và UnitPrice:

            Quantity      UnitPrice
count  541909.000000  541909.000000
mean        9.552250       4.611114
std       218.081158      96.759853
min    -80995.000000  -11062.060000
25%         1.000000       1.250000
50%         3.000000       2.080000
75%        10.000000       4.130000
max     80995.000000   38970.000000


## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [10]:
# A.4. Create a new column
clean_df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
clean_df['Sales'] = clean_df['Quantity'] * clean_df['UnitPrice']
print('Kích thước sau khi làm sạch:', clean_df.shape)
display(clean_df[['Quantity', 'UnitPrice', 'Sales']].head())

Kích thước sau khi làm sạch: (530104, 9)


,Quantity,UnitPrice,Sales
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [12]:
# Group 1 — Central Tendency
num_cols = ['Quantity', 'UnitPrice', 'Sales']
summary = clean_df[num_cols].agg(['mean', 'median']).T
summary.columns = ['mean', 'median']
print(summary)
print('\nMode:')
for col in num_cols:
    print(col, ':', clean_df[col].mode().iloc[0])

                mean  median
Quantity   10.542037    3.00
UnitPrice   3.907625    2.08
Sales      20.121871    9.90

Mode:
Quantity : 1
UnitPrice : 1.25
Sales : 15.0


## Group 2 — Dispersion

In [13]:
# Group 2 — Dispersion
num_cols = ['Quantity', 'UnitPrice', 'Sales']
print(clean_df[num_cols].agg(['min', 'max', 'std', 'var']).T)
print('\nIQR:')
for col in num_cols:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    print(col, ':', round(q3 - q1, 2))

             min        max         std           var
Quantity   1.000   80995.00  155.524124  24187.752994
UnitPrice  0.001   13541.33   35.915681   1289.936149
Sales      0.001  168469.60  270.356743  73092.768604

IQR:
Quantity : 9.0
UnitPrice : 2.88
Sales : 13.95


## Group 3 — Location and Shape

In [14]:
# Group 3 — Location and Shape
for col in ['Quantity', 'UnitPrice', 'Sales']:
    print('\n===', col, '===')
    print('Q1:', clean_df[col].quantile(0.25))
    print('Median:', clean_df[col].median())
    print('Q3:', clean_df[col].quantile(0.75))
    print('Skewness:', round(clean_df[col].skew(), 4))
    print('Kurtosis:', round(clean_df[col].kurt(), 4))


=== Quantity ===
Q1: 1.0
Median: 3.0
Q3: 10.0
Skewness: 471.7277
Kurtosis: 236462.3428

=== UnitPrice ===
Q1: 1.25
Median: 2.08
Q3: 4.13
Skewness: 206.0876
Kurtosis: 62483.1427

=== Sales ===
Q1: 3.75
Median: 9.9
Q3: 17.700000000000003
Skewness: 506.706
Kurtosis: 297651.661


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [15]:
# Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?
country_sales = clean_df.groupby('Country')['Sales'].sum().sort_values(ascending=False)
total_sales = country_sales.sum()
share = country_sales / total_sales * 100
print('Doanh thu theo quốc gia:')
display(country_sales.head())
print('\nTỷ trọng doanh thu:')
display(share.head())
print('\nQuốc gia đóng góp cao nhất:', country_sales.idxmax())
print('Tỷ lệ %:', round(share.max(), 2), '%')

Doanh thu theo quốc gia:


Country
United Kingdom    9025222.084
Netherlands        285446.340
EIRE               283453.960
Germany            228867.140
France             209715.110
Name: Sales, dtype: float64


Tỷ trọng doanh thu:


Country
United Kingdom    84.611315
Netherlands        2.676055
EIRE               2.657376
Germany            2.145626
France             1.966076
Name: Sales, dtype: float64


Quốc gia đóng góp cao nhất: United Kingdom
Tỷ lệ %: 84.61 %


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [17]:
# Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?
product_sales = clean_df.groupby('Description')['Sales'].sum().sort_values(ascending=False)
print('Top sản phẩm theo doanh thu:')
display(product_sales.head(10))
print('\nSản phẩm bán chạy nhất:', product_sales.index[0])
print('Doanh thu:', round(product_sales.iloc[0], 2))

Top sản phẩm theo doanh thu:


Description
DOTCOM POSTAGE                        206248.77
REGENCY CAKESTAND 3 TIER              174484.74
PAPER CRAFT , LITTLE BIRDIE           168469.60
WHITE HANGING HEART T-LIGHT HOLDER    106292.77
PARTY BUNTING                          99504.33
JUMBO BAG RED RETROSPOT                94340.05
MEDIUM CERAMIC TOP STORAGE JAR         81700.92
Manual                                 78112.82
POSTAGE                                78101.88
RABBIT NIGHT LIGHT                     66964.99
Name: Sales, dtype: float64


Sản phẩm bán chạy nhất: DOTCOM POSTAGE
Doanh thu: 206248.77


## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [18]:
# Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?
clean_df['Month'] = clean_df['InvoiceDate'].dt.to_period('M').astype(str)
monthly_sales = clean_df.groupby('Month')['Sales'].sum().sort_index()
print(monthly_sales)
print('\nTháng có doanh thu cao nhất:', monthly_sales.idxmax())
print('Doanh thu cao nhất:', round(monthly_sales.max(), 2))

Month
2010-12     823746.140
2011-01     691364.560
2011-02     523631.890
2011-03     717639.360
2011-04     537808.621
2011-05     770536.020
2011-06     761739.900
2011-07     719221.191
2011-08     759138.380
2011-09    1058590.172
2011-10    1154979.300
2011-11    1509496.330
2011-12     638792.680
Name: Sales, dtype: float64

Tháng có doanh thu cao nhất: 2011-11
Doanh thu cao nhất: 1509496.33


## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [19]:
# Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?
aov_by_country = clean_df.groupby(['Country', 'InvoiceNo'])['Sales'].sum().groupby(level=0).mean().sort_values(ascending=False)
print('Average Order Value theo quốc gia:')
display(aov_by_country.head(10))
print('\nQuốc gia có AOV cao nhất:', aov_by_country.idxmax())
print('AOV cao nhất:', round(aov_by_country.max(), 2))

Average Order Value theo quốc gia:


Country
Singapore      3039.898571
Netherlands    3036.663191
Australia      2430.198421
Japan          1969.282632
Lebanon        1693.880000
Hong Kong      1426.527273
Brazil         1143.600000
Sweden         1066.064722
Switzerland    1057.220370
Denmark        1053.074444
Name: Sales, dtype: float64


Quốc gia có AOV cao nhất: Singapore
AOV cao nhất: 3039.9


## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [20]:
# Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?
neg_ratio = df.groupby('Country')['Quantity'].apply(lambda x: (x < 0).mean() * 100).sort_values(ascending=False)
print('Tỷ lệ Quantity âm theo quốc gia (%):')
display(neg_ratio.head(10))
print('\nQuốc gia có tỷ lệ trả hàng cao nhất:', neg_ratio.idxmax())
print('Tỷ lệ cao nhất:', round(neg_ratio.max(), 2), '%')

Tỷ lệ Quantity âm theo quốc gia (%):


Country
USA               38.487973
Czech Republic    16.666667
Malta             11.811024
Japan             10.335196
Saudi Arabia      10.000000
Australia          5.877681
Italy              5.603985
Bahrain            5.263158
Germany            4.770932
EIRE               3.684724
Name: Quantity, dtype: float64


Quốc gia có tỷ lệ trả hàng cao nhất: USA
Tỷ lệ cao nhất: 38.49 %


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*

Doanh thu của dữ liệu Online Retail tập trung chủ yếu ở Anh, cho thấy thị trường này chiếm phần lớn doanh số và đóng vai trò quan trọng trong hoạt động bán hàng. Các sản phẩm có doanh thu lớn thường là mặt hàng có giá trị hoặc bán chạy nhất, nên doanh nghiệp nên ưu tiên kiểm soát hàng tồn kho cho nhóm sản phẩm này. Doanh số có xu hướng thay đổi theo tháng, cho thấy có yếu tố mùa vụ trong hoạt động bán hàng. Average Order Value khác nhau giữa các quốc gia, nhấn mạnh sự khác biệt về mức chi tiêu bình quân của khách hàng theo khu vực. Bên cạnh đó, tỷ lệ giao dịch âm trong dữ liệu gốc cho thấy có hiện tượng trả hàng hoặc hủy đơn, và mức độ này không đồng đều giữa các quốc gia nên cần được theo dõi thêm.